# 56-bus Admittance Stress Test

This notebook probes how the trained 56-bus RLC-FT controller behaves when the tested line-admittance range exceeds the bound used in the conservative stability certificate. The experiment keeps the original code unchanged: after `env.reset_topo(seed)` samples the usual topology and +/-50% line-reactance perturbation, we further scale the active line admittances by a multiplicative factor, equivalently reducing active line reactances by the same factor.

The test is empirical. It does not extend the formal theorem beyond its prescribed bounds; it checks whether closed-loop voltage recovery degrades gradually when the certified rate becomes more conservative.

In [ ]:
from Environment import *
from DDPG import *
from NN_Module import *
from config import Config

import os
import sys
import json
import gzip
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from numpy import linalg as LA
from loguru import logger

# Keep outputs consistent with the rest of the project.
PROJECT_ROOT = Path(Config.data_path)
CACHE_DIR = PROJECT_ROOT / ".cache" / "notebook_results"
OUTPUT_DIR = PROJECT_ROOT / "images" / "56bus" / "admittance_stress"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pilot settings. Each admittance scaling factor uses the same 100 paired topology/admittance scenarios.
EPISODE_NUM = 100
STEP_NUM = 200
STRESS_FACTORS = [1.0, 1.5, 2.0, 3.0, 5.0]
METHODS_TO_RUN = ["RLC-FT", "Linear", "Safe-DDPG"]
PROGRESS_EVERY = 20

# Theory-side reference used only for labeling/interpretation.
# If b_max is increased by factor eta and kappa is fixed, the certified rate scales as 1/eta.
DESIGN_BMAX_PU = 1.0e3
DESIGN_DMAX = 5
KAPPA = Config.K

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
logger.remove()
logger.add(sys.stderr, level="INFO")
print(f"Using device: {device}")
print(f"Saving pilot outputs to: {OUTPUT_DIR}")

In [ ]:
def _cache_path(name):
    return CACHE_DIR / f"{name}.pkl.gz"


def save_result(name, data):
    path = _cache_path(name)
    with gzip.open(path, "wb") as f:
        pickle.dump({"data": data}, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved result: {path}")
    return path


def load_result(name):
    path = _cache_path(name)
    if not path.exists():
        raise FileNotFoundError(f"No cached result found for {name}: {path}")
    with gzip.open(path, "rb") as f:
        payload = pickle.load(f)
    print(f"Loaded result: {path}")
    return payload["data"]


def first_existing(*paths):
    for path in paths:
        p = Path(path)
        if p.exists():
            return p
    raise FileNotFoundError("None of these paths exists: " + "; ".join(str(p) for p in paths))

In [ ]:
# Create the 56-bus testing environment and load the same trained models used in test_56bus_performance.ipynb.
agent_num = 5
injection_bus = np.array([18, 21, 30, 45, 53]) - 1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)

agent_policy_net = []
safe_agent_net = []

for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=256)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=2048).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    rlcft_model = first_existing(
        Path(Config.data_path) / f"check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth",
        PROJECT_ROOT / f"check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth",
    )
    agent_policy_net[i].load_state_dict(torch.load(rlcft_model, map_location=device))
    agent_policy_net[i].eval()

for i in range(agent_num):
    safe_model = first_existing(
        Path("D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg") / f"policy_net_checkpoint_a{i}.pth",
        PROJECT_ROOT / "saved_models" / "stable_ddpg" / f"policy_net_checkpoint_a{i}.pth",
    )
    safe_agent_net[i].load_state_dict(torch.load(safe_model, map_location=device))
    safe_agent_net[i].eval()

print("Models loaded.")

In [ ]:
def closed_line_mask(env):
    """Return a boolean mask for lines that are not opened by switch status."""
    mask = np.ones(len(env.network.line), dtype=bool)
    for _, sw in env.network.switch.iterrows():
        if sw.et == "l" and not bool(sw.closed):
            mask[int(sw.element)] = False
    return mask


def apply_admittance_stress(env, topology, stress_factor):
    """Increase active line admittances after env.reset_topo without modifying Environment.py.

    env.reset_topo already samples x_ij in [0.5, 1.5] times the nominal value and
    stores topology entries as approximately 1/x_ij, while open switched lines are
    replaced by near-zero random entries. We preserve those near-zero open-line
    entries and multiply only the active-line topology entries by stress_factor.
    """
    topology_arr = np.asarray(topology, dtype=float).copy()
    if stress_factor <= 0:
        raise ValueError("stress_factor must be positive")

    if stress_factor != 1.0:
        mask = closed_line_mask(env)
        current_x = env.network.line["x_ohm_per_km"].astype(float).to_numpy().copy()
        stressed_x = current_x.copy()
        stressed_x[mask] = stressed_x[mask] / stress_factor
        env.network.line.loc[:, "x_ohm_per_km"] = stressed_x
        topology_arr[mask] = topology_arr[mask] * stress_factor

        pp.runpp(env.network, algorithm="bfsw")
        env.state = env.network.res_bus.iloc[env.injection_bus].vm_pu.to_numpy()
        env.topology = topology_arr

    return topology_arr


def rlcft_action(state, topology_tensor, last_action):
    action = []
    for i in range(agent_num):
        state_tensor = torch.tensor(state[i].reshape(1,), dtype=torch.float32, device=device).unsqueeze(0)
        action_agent = agent_policy_net[i](state_tensor, topology_tensor)
        action_agent = 0.7 * action_agent.detach().cpu().numpy()[0]
        action.append(action_agent)
    return last_action - np.asarray(action)


def linear_action(state, last_action):
    state1 = np.asarray(state - env.vmax)
    state2 = np.asarray(env.vmin - state)
    d_v = (np.maximum(state1, 0) - np.maximum(state2, 0)).reshape((agent_num, 1))
    return last_action - 10 * d_v


def safe_ddpg_action(state, last_action):
    action = []
    for i in range(agent_num):
        state_tensor = torch.tensor([state[i]], dtype=torch.float32, device=device).reshape(1, 1)
        action_agent = safe_agent_net[i](state_tensor).detach().cpu().numpy()[0]
        action.append(action_agent)
    return last_action - 5 * np.asarray(action).reshape((agent_num, 1))


def policy_action(method, state, topology_tensor, last_action):
    if method == "RLC-FT":
        return rlcft_action(state, topology_tensor, last_action)
    if method == "Linear":
        return linear_action(state, last_action)
    if method == "Safe-DDPG":
        return safe_ddpg_action(state, last_action)
    raise ValueError(f"Unknown method: {method}")

In [ ]:
def run_one_episode(method, seed, stress_factor):
    try:
        state, topology, scenario = env.reset_topo(seed=seed)
        topology = apply_admittance_stress(env, topology, stress_factor)
        state = env.state.copy()
    except Exception as exc:
        return {
            "method": method,
            "seed": seed,
            "stress_factor": stress_factor,
            "certified_rate_ratio": 1.0 / stress_factor,
            "scenario": np.nan,
            "status": "initial_powerflow_fail",
            "recovery_time": STEP_NUM,
            "control_cost": np.nan,
            "objective_cost": np.nan,
            "fail_step": 0,
            "message": repr(exc),
        }

    topology_tensor = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
    last_action = np.zeros((agent_num, 1))
    episode_reward = 0.0
    episode_control = 0.0
    costs = []
    done = False
    abnormal_stop = False
    fail_step = None
    message = ""

    for step in range(STEP_NUM):
        action = policy_action(method, state, topology_tensor, last_action)
        last_action = np.copy(action)

        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged as exc:
            abnormal_stop = True
            fail_step = step
            message = repr(exc)
            break

        if np.min(next_state) < 0.75 or np.max(next_state) > 1.25:
            abnormal_stop = True
            fail_step = step
            message = f"voltage exceeded guardrail: min={np.min(next_state):.3f}, max={np.max(next_state):.3f}"
            break

        if done:
            break

        state = next_state
        episode_reward += reward
        costs.append(-reward)
        episode_control += LA.norm(action, 2)

    if done:
        status = "recovered"
        recovery_time = step
    elif abnormal_stop:
        status = "abnormal_stop"
        recovery_time = STEP_NUM
    else:
        status = "horizon_reached"
        recovery_time = STEP_NUM

    return {
        "method": method,
        "seed": seed,
        "stress_factor": stress_factor,
        "certified_rate_ratio": 1.0 / stress_factor,
        "scenario": int(scenario),
        "status": status,
        "recovery_time": float(recovery_time),
        "control_cost": float(episode_control),
        "objective_cost": float(np.sum(costs)),
        "fail_step": fail_step,
        "message": message,
    }


records = []
for stress_factor in STRESS_FACTORS:
    for method in METHODS_TO_RUN:
        for episode in range(EPISODE_NUM):
            records.append(run_one_episode(method, episode, stress_factor))
            if (episode + 1) % PROGRESS_EVERY == 0 or (episode + 1) == EPISODE_NUM:
                logger.info(
                    "stress={} method={} progress {}/{}",
                    stress_factor,
                    method,
                    episode + 1,
                    EPISODE_NUM,
                )

stress_df = pd.DataFrame(records)
stress_df.to_csv(OUTPUT_DIR / "56bus_admittance_stress_raw_100.csv", index=False)
save_result("56bus_admittance_stress_100", {"records": records, "settings": {
    "episode_num": EPISODE_NUM,
    "step_num": STEP_NUM,
    "stress_factors": STRESS_FACTORS,
    "methods": METHODS_TO_RUN,
    "design_bmax_pu": DESIGN_BMAX_PU,
    "design_dmax": DESIGN_DMAX,
    "kappa": KAPPA,
}})
stress_df.head()

In [ ]:
def q25(x):
    return np.quantile(x, 0.25)


def q75(x):
    return np.quantile(x, 0.75)

summary = (
    stress_df
    .groupby(["stress_factor", "certified_rate_ratio", "method"], as_index=False)
    .agg(
        recovery_median=("recovery_time", "median"),
        recovery_q25=("recovery_time", q25),
        recovery_q75=("recovery_time", q75),
        control_median=("control_cost", "median"),
        control_q25=("control_cost", q25),
        control_q75=("control_cost", q75),
        objective_median=("objective_cost", "median"),
        objective_q25=("objective_cost", q25),
        objective_q75=("objective_cost", q75),
        recovered_within_horizon=("status", lambda s: float(np.mean(np.asarray(s) == "recovered"))),
        abnormal_stop_rate=("status", lambda s: float(np.mean(np.asarray(s) == "abnormal_stop"))),
        n=("seed", "count"),
    )
)
summary.to_csv(OUTPUT_DIR / "56bus_admittance_stress_summary_100.csv", index=False)
summary

In [ ]:
colors = {
    "Linear": "#BDBDBD",
    "Safe-DDPG": "#5B8DB8",
    "RLC-FT": "#1B9E77",
}

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Voltage recovery time",
        "Total objective cost",
        "Recovered within horizon",
    ],
    horizontal_spacing=0.10,
)

for method in METHODS_TO_RUN:
    df = summary[summary["method"] == method].sort_values("stress_factor")
    x = df["stress_factor"]

    fig.add_trace(
        go.Scatter(
            x=x,
            y=df["recovery_median"],
            error_y=dict(
                type="data",
                symmetric=False,
                array=df["recovery_q75"] - df["recovery_median"],
                arrayminus=df["recovery_median"] - df["recovery_q25"],
                thickness=1.6,
                width=4,
            ),
            mode="lines+markers",
            name=method,
            legendgroup=method,
            line=dict(color=colors.get(method), width=2.5),
            marker=dict(size=8),
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=df["objective_median"],
            error_y=dict(
                type="data",
                symmetric=False,
                array=df["objective_q75"] - df["objective_median"],
                arrayminus=df["objective_median"] - df["objective_q25"],
                thickness=1.6,
                width=4,
            ),
            mode="lines+markers",
            name=method,
            legendgroup=method,
            line=dict(color=colors.get(method), width=2.5),
            marker=dict(size=8),
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=100.0 * df["recovered_within_horizon"],
            mode="lines+markers",
            name=method,
            legendgroup=method,
            line=dict(color=colors.get(method), width=2.5),
            marker=dict(size=8),
            showlegend=False,
        ),
        row=1,
        col=3,
    )

fig.update_layout(
    width=1250,
    height=450,
    font=dict(family="Arial, sans-serif", size=16),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(x=1.02, y=1.0, xanchor="left", yanchor="top"),
    margin=dict(l=75, r=150, t=70, b=70),
)

fig.update_xaxes(title_text="Admittance scaling factor", tickmode="array", tickvals=STRESS_FACTORS, ticktext=[str(x) for x in STRESS_FACTORS], showline=True, linecolor="black", linewidth=1.2)
fig.update_yaxes(title_text="Steps, median [IQR]", row=1, col=1, showline=True, linecolor="black", linewidth=1.2, gridcolor="rgba(0,0,0,0.12)")
fig.update_yaxes(title_text="Objective cost, median [IQR]", row=1, col=2, showline=True, linecolor="black", linewidth=1.2, gridcolor="rgba(0,0,0,0.12)")
fig.update_yaxes(title_text="%", row=1, col=3, range=[0, 105], showline=True, linecolor="black", linewidth=1.2, gridcolor="rgba(0,0,0,0.12)")

if "ipykernel" in sys.modules:
    fig.show()
else:
    print("Skipped interactive Plotly display outside a notebook kernel.")

try:
    fig.write_image(OUTPUT_DIR / "56bus_admittance_stress_summary_100.png", width=1250, height=450, scale=2)
    fig.write_image(OUTPUT_DIR / "56bus_admittance_stress_summary_100.pdf", width=1250, height=450)
    print(f"Saved figure to {OUTPUT_DIR}")
except Exception as exc:
    print(f"Figure display worked, but static export failed: {exc}")

## Reading the pilot

- `stress_factor = 1.0` reproduces the usual +/-50% line-reactance perturbation range.
- Larger values reduce active line reactances after topology sampling, so the tested line admittances exceed the design envelope.
- The formal guarantee at the original rate is not claimed outside the prescribed bound. With fixed `kappa`, the certified rate scales approximately as `1 / stress_factor`.
- The most useful outcome is graceful degradation: recovery time and control action may increase as the admittance scaling factor grows, while RLC-FT should ideally remain within the 200-step horizon for moderate out-of-bound stress.